In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import joblib
from sklearn.metrics import roc_curve

In [9]:
model = joblib.load("../models/fraud_detection_model.pkl")
preprocessor = joblib.load("../models/fraud_detection_preprocessor.pkl")

In [12]:
def predict_transaction(
    step,
    transaction_type,
    amount,
    oldbalanceOrg,
    newbalanceOrig,
    oldbalanceDest,
    newbalanceDest,
    isFlaggedFraud
):
    transaction = pd.DataFrame([{
        "step": step,
        "type": transaction_type,
        "amount": amount,
        "oldbalanceOrg": oldbalanceOrg,
        "newbalanceOrig": newbalanceOrig,
        "oldbalanceDest": oldbalanceDest,
        "newbalanceDest": newbalanceDest,
        "isFlaggedFraud": isFlaggedFraud
    }])

    # Preprocess transaction
    transaction_processed = preprocessor.transform(transaction)

    # Fraud probability
    fraud_probability = model.predict_proba(
        transaction_processed
    )[0, 1]

    # Prediction
    prediction = model.predict(
        transaction_processed
    )[0]

    # Risk level
    if fraud_probability < 0.30:
        risk_level = "Low"
    elif fraud_probability < 0.70:
        risk_level = "Medium"
    else:
        risk_level = "High"

    result = {
        "Prediction": "Fraud" if prediction == 1 else "Legitimate",
        "Fraud Probability": f"{fraud_probability * 100:.2f}%",
        "Risk Level": risk_level
    }

    return result

In [11]:
result = predict_transaction(
    step=1,
    transaction_type="TRANSFER",
    amount=500000,
    oldbalanceOrg=500000,
    newbalanceOrig=0,
    oldbalanceDest=0,
    newbalanceDest=500000,
    isFlaggedFraud=0
)

print(result)

{'Prediction': 'Legitimate', 'Fraud Probability': '49.77%', 'Risk Level': 'Medium'}


## Final Model Evaluation

The final Logistic Regression + SMOTE model achieved:

- Precision: 67.44%
- Recall: 59.89%
- F1-Score: 63.44%
- ROC-AUC: 99.23%

The model detected 984 out of 1,643 fraudulent transactions in the test
set, while 659 fraudulent transactions were missed.

The model provides a strong ROC-AUC and improved fraud recall compared with
the baseline Logistic Regression model.